In [ ]:
# Import all the tools we need
import scanpy as sc        # Main scRNA-seq analysis library
import pandas as pd        # For working with tables
import numpy as np         # For math operations
import matplotlib.pyplot as plt  # For making plots
import seaborn as sns      # For nicer-looking plots
import scrublet as scr     # For detecting doublets (bad data)

# Scanpy display settings
sc.settings.verbosity = 3  # Show detailed progress messages
sc.settings.set_figure_params(dpi=100, facecolor='white')

print("All libraries imported successfully!")

In [ ]:
# Load the CellRanger count matrix
# Reads 3 files: matrix.mtx.gz, barcodes.tsv.gz, features.tsv.gz
adata = sc.read_10x_mtx(
    '/home/jupyter/data/pbmc_1k/filtered_feature_bc_matrix/',
    var_names='gene_symbols',  # Use gene names rather than Ensembl IDs
    cache=True                 # Cache so reloading is faster
)

# Make sure all gene names are unique
adata.var_names_make_unique()

# Print a summary
print(adata)

In [ ]:
# Flag mitochondrial genes
# High mitochondrial gene % = dying or damaged cell
# Human mitochondrial genes all start with "MT-"
adata.var['mt'] = adata.var_names.str.startswith('MT-')

# Calculate QC metrics for every cell
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

# Plot the 3 key QC metrics as violin plots
# n_genes_by_counts = how many genes were detected in each cell
# total_counts      = total RNA molecules detected per cell
# pct_counts_mt     = % of RNA from mitochondrial genes
sc.pl.violin(
    adata,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    jitter=0.4,
    multi_panel=True
)

In [ ]:
print(f"Before filtering: {adata.n_obs} cells, {adata.n_vars} genes")

# Remove cells with too few genes (likely empty droplets)
adata = adata[adata.obs.n_genes_by_counts > 200, :]

# Remove cells with too many genes (likely two cells in one droplet)
adata = adata[adata.obs.n_genes_by_counts < 6000, :]

# Remove cells with high mitochondrial % (likely dying cells)
adata = adata[adata.obs.pct_counts_mt < 20, :]

# Remove genes detected in fewer than 3 cells (too rare to be useful)
sc.pp.filter_genes(adata, min_cells=3)

print(f"After filtering: {adata.n_obs} cells, {adata.n_vars} genes")

In [ ]:
# Run Scrublet
scrub = scr.Scrublet(adata.X)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

# Add results to our data object
adata.obs['doublet_score'] = doublet_scores
adata.obs['predicted_doublet'] = predicted_doublets

# Plot the score distribution — real cells and doublets should form 2 separate peaks
scrub.plot_histogram()
plt.show()

# Remove predicted doublets
adata = adata[~adata.obs['predicted_doublet'], :]
print(f"Cells after doublet removal: {adata.n_obs}")

In [ ]:
# Save a copy of the raw counts — needed for differential expression later
adata.raw = adata

# Normalize: scale every cell to 10,000 total counts
sc.pp.normalize_total(adata, target_sum=1e4)

# Log-transform: log(x+1) — reduces the effect of very highly expressed genes
sc.pp.log1p(adata)

print("Normalization complete!")

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    min_mean=0.0125,
    max_mean=3,
    min_disp=0.5
)

# Plot — highlighted dots are selected genes
sc.pl.highly_variable_genes(adata)

# Keep only the highly variable genes going forward
adata = adata[:, adata.var.highly_variable]
print(f"Highly variable genes selected: {adata.n_vars}")

In [ ]:
# Remove technical effects of total counts and mitochondrial %
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

# Scale to unit variance, clip extreme outliers at 10
sc.pp.scale(adata, max_value=10)

print("Scaling complete!")

In [ ]:
# Run PCA
sc.tl.pca(adata, svd_solver='arpack', n_comps=50)

# Elbow plot — look for where the curve flattens out ("the elbow")
# The elbow tells you how many PCs to use in the next step
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
# n_pcs=30: use the first 30 principal components
# Adjust based on your elbow plot from Step 16
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=30)

print("Neighborhood graph built!")

In [ ]:
# Compute UMAP
sc.tl.umap(adata)

# Plot colored by QC metrics to make sure data looks clean
sc.pl.umap(
    adata,
    color=['total_counts', 'pct_counts_mt', 'n_genes_by_counts'],
    title=['Total Counts', '% Mitochondrial', 'Genes Detected']
)

In [ ]:
# Leiden clustering
# resolution controls number of clusters:
#   lower (0.3) = fewer bigger clusters
#   higher (1.0) = more smaller clusters
# Start with 0.5
sc.tl.leiden(adata, resolution=0.5)

# Plot clusters on UMAP
sc.pl.umap(
    adata,
    color='leiden',
    legend_loc='on data',
    title='Cell Clusters (Leiden)'
)

In [ ]:
# Find marker genes using the Wilcoxon rank-sum test
sc.tl.rank_genes_groups(
    adata,
    groupby='leiden',
    method='wilcoxon',
    use_raw=True
)

# Plot top 10 marker genes per cluster
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

In [ ]:
# Export to a table
import os
os.makedirs('/home/jupyter/results', exist_ok=True)

marker_df = sc.get.rank_genes_groups_df(adata, group=None)
print(marker_df.head(20))

# Save to CSV
marker_df.to_csv('/home/jupyter/results/marker_genes.csv', index=False)
print("Marker genes saved!")

In [ ]:
import celltypist
from celltypist import models

# Reload the original count matrix fresh
adata_celltypist = sc.read_10x_mtx(
    '/home/jupyter/data/pbmc_1k/filtered_feature_bc_matrix/',
    var_names='gene_symbols',
    cache=True
)
adata_celltypist.var_names_make_unique()

# Apply the same cell filtering we did earlier
adata_celltypist = adata_celltypist[adata.obs_names, :]

# Normalize exactly the way CellTypist expects
sc.pp.normalize_total(adata_celltypist, target_sum=1e4)
sc.pp.log1p(adata_celltypist)

# Download model (will skip if already downloaded)
models.download_models(model='Immune_All_Low.pkl')
model = models.Model.load(model='Immune_All_Low.pkl')

# Predict cell types
predictions = celltypist.annotate(adata_celltypist, model=model, majority_voting=True)

# Transfer predictions back to main adata
adata.obs['majority_voting'] = predictions.predicted_labels['majority_voting']

# Plot
sc.pl.umap(adata, color='majority_voting', legend_loc='on data', title='Predicted Cell Types')

In [ ]:
# Save the complete analysis object
adata.write('/home/jupyter/results/scrna_analyzed.h5ad')
print("Analysis object saved!")

# Save cell metadata (cluster IDs, cell types, QC metrics)
adata.obs.to_csv('/home/jupyter/results/cell_metadata.csv')

# Save UMAP coordinates
umap_df = pd.DataFrame(
    adata.obsm['X_umap'],
    columns=['UMAP1', 'UMAP2'],
    index=adata.obs_names
)
umap_df.to_csv('/home/jupyter/results/umap_coordinates.csv')

print("All files saved!")